# 02 — Prototype the Silver transforms

Stage 2 is designed but not implemented (`docs/SETUP.md` §0). The two
transforms Silver needs — natural-key dedupe and event-time bars — are cheap
to get wrong in ways that only show up much later, so get the semantics right
here in pandas first, then port them.

Each section ends with the PySpark it maps to. This notebook is where the
*decision* is made; the pipeline is where it gets executed at scale.

In [ ]:
import devlab
from devlab import frames
from devlab.frames import NATURAL_KEY

target = devlab.resolve()
records = devlab.collect(target, limit=20_000, seconds=60.0, offset_reset="earliest")
raw = frames.trades_frame(records)
print(f"{len(raw):,} raw records")
NATURAL_KEY

## 1. Dedupe on the natural key

`(venue, venue_symbol, trade_id)` is the identity of a trade. `trade_id` alone
is **not** unique — it is only unique within a venue, and both venues number
from 1.

Duplicates are expected, not a bug: REST gap repair re-fetches a range that
partly overlaps what the stream already delivered, and archive backfill will
re-emit history that is already in the log. Both are deliberate (see the
Coinbase connector's "best-effort repair" note). Silver absorbs the overlap.

In [ ]:
duplicated = raw[raw.duplicated(subset=NATURAL_KEY, keep=False)]
print(f"{len(duplicated):,} rows share a natural key with another row")
duplicated.sort_values(NATURAL_KEY)[[*NATURAL_KEY, "source", "kafka_offset", "price"]].head(10)

In [ ]:
silver = frames.dedupe(raw)
print(f"{len(raw):,} raw -> {len(silver):,} deduped ({len(raw) - len(silver):,} dropped)")
silver["source"].value_counts()

Check the drop is real rather than an over-collapse: if `trade_id` were being
compared across venues, the count would fall by roughly half.

In [ ]:
frames.frame(
    silver.groupby("venue", observed=True)
    .agg(trades=("trade_id", "count"), unique_ids=("trade_id", "nunique"))
    .reset_index()
    .to_dict("records")
)

**PySpark equivalent**

```python
from pyspark.sql import functions as F, Window

window = Window.partitionBy(*NATURAL_KEY).orderBy("event_ts", "kafka_offset")
silver = (
    bronze
    .withColumn("_rank", F.row_number().over(window))
    .filter(F.col("_rank") == 1)
    .drop("_rank")
)
```

`dropDuplicates(NATURAL_KEY)` is shorter but picks an arbitrary survivor. The
window makes the tie-break explicit, which matters when a `STREAM` row and a
`REST_REPAIR` row disagree.

For streaming, this needs `withWatermark("event_ts", ...)` bounding how late a
repair may arrive — the repair latency is your watermark floor.

## 2. Event-time bars

Built on `event_ts` (exchange time), never `ingest_ts`. Using arrival time
would let a slow consumer reshape the data — replay the same log through a
loaded cluster and you would get different bars, which is exactly the
non-determinism a lakehouse is supposed to eliminate.

Dedupe first. Duplicated trades inflate both volume and VWAP.

In [ ]:
bars = frames.bars(silver, freq="1min")
bars.tail(10)

Sanity check: VWAP must sit within the bar's `low..high`. A weighted mean of
values drawn from that range, with non-negative weights, cannot escape it — so
a real violation means the volume weighting is wrong.

The tolerance is not slop. On a **flat bar** (every trade at one price)
`sum(p*s) / sum(s)` is `p` only up to float64 rounding, and lands roughly one
ulp outside `low..high`. Verified against live data: 6 of 97 bars tripped a
zero-tolerance check, all flat, all off by ~9e-16 — noise, not a defect.

It is also the argument for `decimal(38, 18)` rather than `double` in Spark:
the wire format keeps prices exact as strings, and Silver should not be the
place that quietly stops being exact.

In [ ]:
tolerance = 1e-9 * bars["high"].abs()  # float64 noise on flat bars, not a breach
outside = bars[(bars["vwap"] < bars["low"] - tolerance) | (bars["vwap"] > bars["high"] + tolerance)]
print(f"{len(outside)} bars with VWAP outside low..high (expected 0)")
outside

In [ ]:
btc = bars[bars["instrument_id"] == "BTC-USD"]
if not btc.empty:
    axis = btc.plot(x="event_ts", y=["open", "high", "low", "close", "vwap"], figsize=(11, 4))
    axis.set_ylabel("BTC-USD")
    axis.set_xlabel("event time (UTC)")

The default groups by `instrument_id` alone — a **consolidated tape** across
venues. That is a real modelling decision, not a default to accept silently:
it treats a Binance trade and a Coinbase trade in the same instrument as the
same market. Pass `by=["venue", "instrument_id"]` for per-venue bars.

In [ ]:
frames.bars(silver, freq="1min", by=["venue", "instrument_id"]).tail(8)

**PySpark equivalent**

```python
gold = (
    silver
    .groupBy(F.window("event_ts", "1 minute"), "instrument_id")
    .agg(
        F.first("price").alias("open"),
        F.max("price").alias("high"),
        F.min("price").alias("low"),
        F.last("price").alias("close"),
        F.sum("size").alias("volume"),
        F.sum("notional").alias("notional"),
        F.count("*").alias("trades"),
    )
    .withColumn("vwap", F.col("notional") / F.col("volume"))
)
```

Two differences from pandas that will bite:

- `F.first` / `F.last` are **not** ordered unless the input is. `pd.Grouper`
  preserves the sorted index; Spark does not. Order within the window, or use
  `min_by("price", "event_ts")` / `max_by`.
- `price` and `size` are strings on the wire. Cast to `decimal(38, 18)` in
  Spark rather than double — this notebook uses float64 for convenience, but
  Silver should not throw away the precision the wire format preserves.